<a href="https://colab.research.google.com/github/Stubberson/project-collection/blob/main/aalto-thesis/standard_distance.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Standard Distance
I'll try to approximate positional accuracy by first calculating the standard distance for each questionnaire (done in Superset), and then calculating the rate of points outside the standard distance in the questionnaire (implemented here).

I'm using the Haversine formula to calculate distances between two coordinate points. It takes the earth's curvature into account.

---

> I would need to adapt this so that I first do clustering of coordinate points (e.g. DBSCAN) and then calculate the standard distance on each cluster. Treating the whole coordinate point dataset as one cluster does not work.



In [ ]:
# Try DBSCAN for clustering!

In [ ]:
import math

def is_point_in_circle(test_lat, test_lon, center_lat, center_lon, radius_meters):
    """
    Checks if a geographic point is within a circle using the Haversine formula.

    Args:
        test_lat (float): Latitude of the point to check.
        test_lon (float): Longitude of the point to check.
        center_lat (float): Latitude of the circle's center.
        center_lon (float): Longitude of the circle's center.
        radius_meters (float): The circle's radius in meters.

    Returns:
        bool: True if the point is inside or on the edge of the circle, False otherwise.
    """
    # Earth's radius in meters (volumetric mean radius (src: NASA))
    R = 6371000

    # Convert latitude and longitude from degrees to radians
    c_lat_rad = math.radians(center_lat)
    c_lon_rad = math.radians(center_lon)
    lat_rad = math.radians(test_lat)
    lon_rad = math.radians(test_lon)

    # Difference in coordinates
    dlon = lon_rad - c_lon_rad
    dlat = lat_rad - c_lat_rad

    # Haversine formula
    a = math.sin(dlat / 2)**2 + math.cos(c_lat_rad) * math.cos(lat_rad) * math.sin(dlon / 2)**2
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))
    distance = R * c

    # Compare the distance to the radius
    return distance <= radius_meters

In [ ]:
import pandas as pd

df_aggregates = pd.read_csv('/content/spatial_accuracy.csv')
display(df_aggregates.head())

df_points = pd.read_csv('/content/lats_lons_3sa.csv')
display(df_points.head())

In [ ]:
# Initialize counters
points_inside = 0
points_outside = 0

# Iterate through each point in df_points
for index, point_row in df_points.iterrows():
    point_q_id = point_row['q_id']
    point_lat = point_row['lat']
    point_lon = point_row['lon']

    # Find the corresponding aggregate data for the q_id
    aggregate_row = df_aggregates[df_aggregates['q_id'] == point_q_id]

    if not aggregate_row.empty:
        center_lat = aggregate_row['lat_avg'].iloc[0]
        center_lon = aggregate_row['lon_avg'].iloc[0]
        radius_meters = aggregate_row['std_distance_m'].iloc[0]

        # Check if the point is inside the circle
        if is_point_in_circle(point_lat, point_lon, center_lat, center_lon, radius_meters):
            points_inside += 1
        else:
            points_outside += 1
    else:
        print(f"Warning: No aggregate data found for q_id: {point_q_id}")

# Print the results
print(f"Number of points inside the circle: {points_inside}")
print(f"Number of points outside the circle: {points_outside}")
print(f"Rate of points inside the circle: {points_inside / len(df_points)}")